In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import missingno as msno

In [28]:
df=pd.read_csv("C:/Users/adaml/OneDrive/Documents/Data Projects/9 Projects for ITonlinelearning portfolio/googleplaystore.csv")
df.sample(5)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
2782,Etsy: Handmade & Vintage Goods,SHOPPING,4.3,95520,15M,"10,000,000+",Free,0,Teen,Shopping,"August 3, 2018",5.3.1,4.1 and up
3489,Samsung Notes,PRODUCTIVITY,3.9,15368,Varies with device,"100,000,000+",Free,0,Everyone,Productivity,"January 22, 2018",Varies with device,Varies with device
9176,EB Mobile,FAMILY,1.7,1172,5.6M,"10,000+",Free,0,Everyone,Education,"October 9, 2017",1.1.2,4.1 and up
1335,Meditation Studio,HEALTH_AND_FITNESS,4.6,1026,29M,"10,000+",Paid,$3.99,Everyone,Health & Fitness,"May 15, 2018",1.0.6,4.3 and up
4132,F-Sim Space Shuttle,FAMILY,4.4,5427,Varies with device,"100,000+",Paid,$4.99,Everyone,Simulation,"January 28, 2014",Varies with device,Varies with device


### 1. Data Cleaning

In [29]:
#Checking
df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [30]:
#Cleaning the 'Rating' column and other columns containing null values
df.loc[df['Rating']>5, 'Rating'] = np.nan

df['Rating'] = df['Rating'].fillna(df['Rating'].mean())
df.dropna(inplace=True)

In [31]:
#Cleaning the column 'Reviews' and changing it to numeric data type

df.loc[df['Reviews'].str.contains('M'), 'Reviews']=(pd.to_numeric(
    df.loc[df['Reviews'].str.contains('M'), 'Reviews'].str.replace('M',''))*1_000_000).astype('str')

df['Reviews'] = pd.to_numeric(df['Reviews'])


In [33]:
#Identifying number of duplicates in 'App'
df['App'].duplicated(keep=False).sum()

np.int64(1979)

In [34]:
#Dropping duplicated apps but keeping only ones with the greatest number of reviews
df_sorted = df.sort_values(by=['App','Reviews'])

df_sorted.loc[
    df_sorted['App'].duplicated(keep=False) &  ~df_sorted.duplicated(keep = False),
    ['App', 'Reviews']
].head(5)

df_sorted.drop_duplicates(subset=['App'], keep='last', inplace=True)
df = df_sorted

In [35]:
# As 1.9 is a wrong category, we transform it into "Unknown" category
#df.loc[df['Category'] == "1.9", 'Category'] = 'Unknown'

# Replace underscores with whitespaces
df["Category"] = df['Category'].str.replace('_', ' ')

# Capitalize the column
df['Category'] = df['Category'].str.capitalize()

In [36]:
# Cleaning and Converting the 'Installs' column to numeric type
df['Installs']=df['Installs'].str.replace('+','').str.replace(',','')
df['Installs']=pd.to_numeric(df['Installs'])

In [37]:
#Cleaning and converting 'Size' column to numeric (representing bytes)

# Setting 'Varies with device' to 0
df['Size'] = df['Size'].replace('Varies with device', "0").astype(str)

# Transforming M to ~1M bytes
new_value = (pd.to_numeric(
    df.loc[df['Size'].str.contains('M'), 'Size'].str.replace('M', '')
) * (1024 * 1024)).astype(str)
df.loc[df['Size'].str.contains('M'), 'Size'] = new_value

# Transforming `k` to ~1k bytes
new_value = (pd.to_numeric(
    df.loc[df['Size'].str.contains('k'), 'Size'].str.replace('k', '')
) * 1024).astype(str)
df.loc[df['Size'].str.contains('k'), 'Size'] = new_value

# Getting rid of `+` and `,`
df['Size'] = df['Size'].str.replace('+', '')
df['Size'] = df['Size'].str.replace(',', '')

# Finally transforming 'Size' to numeric:
df['Size'] = pd.to_numeric(df['Size'])

In [38]:
#Cleaning and converting 'Price' column to numeric
df.loc[df['Price'] == 'Free', 'Price'] = "0"
df['Price'] = df['Price'].str.replace('$', '').str.replace(',', '.')
df['Price'] = pd.to_numeric(df['Price'])

In [39]:
#Creating Auxiliary 'Distribution' column to show 'Free'/'Paid' values depending on the app's price.
df['Distribution'] = 'Free'
df.loc[df['Price'] > 0, 'Distribution'] = 'Paid'

### 2. Analysis

##### Answering Questions:
1. Which app has the most reviews?
2. What Category has the highest number of apps uploaded to the store?
3. To which category belongs the most expensive app?
4. What's the name of the most expensive game?
5. Which is the most popular Finance App?
6. What Teen Game has the most Reviews?
7. Which is the free game with the most reviews?
8. How many TiB (tebibytes) were transferred (overall) for the most popular Lifestyle app?

In [41]:
#1. Answer = 'Facebook'
df[['App', 'Reviews']].sort_values(by=['Reviews'], ascending=False).head()

,App,Reviews
2544,Facebook,78158306
381,WhatsApp Messenger,69119316
2604,Instagram,66577446
382,Messenger – Text and Video Chat for Free,56646578
1879,Clash of Clans,44893888


In [42]:
#2. Answer = 'Family'
df['Category'].value_counts().head()

Category
Family      1874
Game         945
Tools        827
Business     420
Medical      395
Name: count, dtype: int64

In [43]:
#3. Answer = 'Lifestyle'
df[['Category','Price']].sort_values(by=['Price'], ascending=False).head()

,Category,Price
4367,Lifestyle,400.00
5369,Finance,399.99
5359,Finance,399.99
5358,Finance,399.99
5362,Family,399.99


In [44]:
#4. Answer = 'The World Ends With You'
df.loc[df['Category']=='Game'].sort_values(by=['Price'], ascending=False).head(1)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Distribution
4203,The World Ends With You,Game,4.6,4108,13631488.0,10000,Paid,17.99,Everyone 10+,Arcade,"December 14, 2015",1.0.4,4.0 and up,Paid


In [45]:
#5. Answer = 'Google Pay'
df.loc[df['Category']=='Finance'].sort_values(by=['Installs'], ascending=False).head(1)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Distribution
5601,Google Pay,Finance,4.2,348132,0.0,100000000,Free,0.0,Everyone,Finance,"July 26, 2018",2.70.206190089,Varies with device,Free


In [47]:
#6. Answer = 'Asphalt 8: Airborne'
df.loc[(df['Category']=='Game') & (df['Content Rating']=='Teen')].sort_values(by=['Reviews'], ascending=False).head(1)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Distribution
3912,Asphalt 8: Airborne,Game,4.5,8389714,96468992.0,100000000,Free,0.0,Teen,Racing,"July 4, 2018",3.7.1a,4.0.3 and up,Free
